# Summary

Create text files from previously crawls.

In [5]:
import os, sys
import re
import json
import jsonschema

import pandas as pd
from tqdm import tqdm

utils_path = "../../interface/utils"
sys.path.insert(0, utils_path)
from authentication import ApiAuthentication
import gcp_tools as gct

# Set environment variables
dotenv_path = "../../data/environment"
api_configs = ApiAuthentication(dotenv_path=dotenv_path)
api_configs.set_environ_variables()


## Read the crawl configuration file

In [8]:
cp_filename = "crawl_configuraton.csv"
cp_path = "../data/crawler_params"

df_cp = pd.read_csv(os.path.join(cp_path, cp_filename))

df_cp


,seed_url,storage_folder,organization,about,crawl_urls,dont_crawl_urls,crawl_depth,crawl_width
0,https://en.wikipedia.org/wiki/California_Commu...,wikipedia,Wikiepedia,Wikipedia is a free online encyclopedia that a...,https://en.wikipedia.org/wiki/California_Commu...,https://en.wikipedia.org/wiki/File:CDI-Seal-Co...,1,402
1,https://www.cccco.edu,cccco,California Community Colleges,Informational site for all California communit...,https://www.cccco.edu/About-Us/Chancellors-Off...,NaN,1,376
2,https://www.ccleague.org,ccleague,Community College League of California,The Community College League of California is ...,https://www.ccleague.org/resources/ccc-budget-...,NaN,1,375
3,https://cccaoe.org/,cccaoe,California Community College Association For O...,"CCCAOE is a state-wide, membership-driven orga...",https://cccaoe.org/about-cccaoe/#tab-san-franc...,NaN,1,1284
4,https://lao.ca.gov/Publications/,lao,Legislative Analyst's Office,The California Legislature's nonpartisan fisca...,https://lao.ca.gov/Publications/Report/4531;ht...,NaN,1,7
...,...,...,...,...,...,...,...,...
192,https://go.boarddocs.com/ca/whccd/Board.nsf/Pu...,whccd_brd,NaN,NaN,NaN,NaN,10,30
193,https://go.boarddocs.com/ca/vvc/Board.nsf/Public,vvc_brd,NaN,NaN,NaN,NaN,10,30
194,https://go.boarddocs.com/ca/wvm/Board.nsf/public,wvm_brd,NaN,NaN,NaN,NaN,10,30
195,https://go.boarddocs.com/ca/yosemite/Board.nsf...,yosemite_brd,NaN,NaN,NaN,NaN,10,30


## Get a list of files in the GCP bucket

In [9]:
# Get a list of contents in a GCP bucket
gcs_bucket_name = "ccc-crawl_data_xb"
bcontents = gct.gcp_list_bucket(gcp_project_id=os.environ["GOOGLE_CLOUD_PROJECT"],
                                gcs_bucket_name=gcs_bucket_name,
                                path="")

# Pull out the csv files
cfiles = []
for bc in bcontents:
    fp = os.path.split(bc)
    ps = fp[0].split("/")

    source = ps[1] if len(ps) > 1 else ""
    file_name = fp[1] if fp[1].find(".csv") >= 0 else ""

    if len(file_name) > 0:
        cfiles.append(dict(source=source,
                           file_name=file_name,
                           path=fp[0]))

# Create a dataframe with a list of files and their paths
dfc = pd.DataFrame(data=cfiles)
dfc.head()


,source,file_name,path
0,aacc,wwwaaccncheedu_2025May01_1.csv,crawl_data/aacc/webpages_pdfs
1,aacc,wwwaaccncheedu_2025May01_2.csv,crawl_data/aacc/webpages_pdfs
2,aacc,wwwaaccncheedu_2025May01_3.csv,crawl_data/aacc/webpages_pdfs
3,aacc,wwwaaccncheedu_2025May01_4.csv,crawl_data/aacc/webpages_pdfs
4,aacc,wwwaaccncheedu_2025May01_5.csv,crawl_data/aacc/webpages_pdfs


In [42]:
def clean_ptag_texts(ptag_texts: list):
    '''
    Method to clean the ptag text. This method takes a list of ptag texts and
    returns a single string of cleaned ptag texts joined together.
    '''

    # remove unwanted characters
    pats = [r"\n|\xa0", r"\s+", r"\[\d+]",  r"\[…]", r"\s{2,}", r"\||>|/"]

    ptag_text = " ".join(ptag_texts)
    for pat in pats:
        ptag_text = re.sub(pat, " ", ptag_text)

    ptag_text = ptag_text.strip()

    return ptag_text

def create_json_lines_file(data, filename):
    """
    Creates a JSON Lines file from a list of Python objects.

    Args:
        data: A list of Python objects to be written to the file.
        filename: The name of the file to be created.
    """
    with open(filename, 'w') as outfile:
        for item in data:
            json_string = json.dumps(item)
            outfile.write(json_string + '\n')

local_output_path = "../data/jsonl_files"

sources = ['aacc', 'calmatters', 'cccaoe', 'cccco', 'ccleague', 'ccreview', 'columbia',
           'ecs', 'ican', 'lao', 'nsc', 'wikipedia']

################### Handle ccreview
sources = ['aacc', 'calmatters', 'cccaoe', 'cccco', 'ccleague', 'columbia', 'ecs', 'ican', 'lao',
           'nsc', 'wikipedia']

gcs_path = "crawl_data/jsonl_files"

# For each source
for source in tqdm(sources):

    # Get the organization and about information
    mask = df_cp["storage_folder"] == source
    idxcp0 = df_cp[mask].index[0]
    organization = df_cp.loc[idxcp0,"organization"]
    about = df_cp.loc[idxcp0,"about"]
    source_index = df_cp.loc[idxcp0,"storage_folder"]

    # Filter GCP buc ket file listing to just this source
    mask = dfc["source"] == source
    idx0 = dfc[mask].index[0]
    fn_base = dfc.loc[idx0, "file_name"]
    path = dfc.loc[idx0, "path"]
    path = "{}/text_files".format(path[:path.find("/")])
    text_jsonl_filename = "{}_text.jsonl".format(fn_base[:fn_base.rfind("_")])

    # Get the crawl results for this source
    dfs = []
    for idx in dfc[mask].index:
        df = gct.read_csv_file_into_pandas(gcs_project_id=os.environ["GOOGLE_CLOUD_PROJECT"],
                                           gcs_bucket_name=gcs_bucket_name,
                                           gcs_directory=dfc.loc[idx, "path"],
                                           file_name=dfc.loc[idx, "file_name"],
                                           exact=False)
        dfs.append(df)

    # Create a single dataframe
    df = pd.concat(dfs)
    df = df.reset_index(drop=True)

    # Eliminate columns with too little text
    ########################### Handle case with divtag_text
    df["ptag_text_len"] = df["ptag_text"].str.len()

    text_len_min = 249
    mask = df["ptag_text_len"] > text_len_min
    df = df[mask]
    df = df.reset_index(drop=True)

    # Ensure desired columns are string
    for col in ["seed_url", "page_title", "page_url", "crawl_time"]:
        df[col] = df[col].astype("str")

    # Fill NAs
    df = df.fillna(value="")

    # # Create a text string with all crawled text
    txt_lines = []
    for i, idx in enumerate(df.index):

        if "divtag_text" in df.columns:
            content=clean_ptag_texts(ptag_texts=[df.loc[idx, "ptag_text"], df.loc[idx, "divtag_text"]])
        else:
            content=clean_ptag_texts(ptag_texts=[df.loc[idx, "ptag_text"]])

        # Add a to a list of dictionaries to create JSONL files
        txt_lines.append(dict(title= "" if df.loc[idx, "page_title"] == "nan" else df.loc[idx, "page_title"],
                              description="web page text",
                              source_index=source_index,
                              media_type="text",
                              language_code="en-US",
                              categories=["education > text"],
                              uri=df.loc[idx, "page_url"],
                              country_origin="US",
                              content_index=i,
                              transcript=content,
                              organizations=[dict(name=organization,
                                                  role=about,
                                                  uri=df.loc[idx, "seed_url"])],
                              available_time = df.loc[idx, "crawl_time"]
                          )
         )

    # Save a local jsonl file
    create_json_lines_file(data=txt_lines,
                           filename=os.path.join(local_output_path, text_jsonl_filename))

    # Save descriptions dataframe in a CSV file on GCP
    # gct.save_file_to_bucket(gcs_project_id=os.environ["GOOGLE_CLOUD_PROJECT"],
    #                         gcs_bucket_name=gcs_bucket_name,
    #                         gcs_directory=path,
    #                         file_name=text_txt_filename,
    #                         content=crawled_txt)


100%|██████████| 11/11 [01:23<00:00,  7.56s/it]


## Validate JSON against the GCS data store JSON format

In [11]:
def read_jsonl_file_json(file_path):
    """Reads a JSONL file using the json library and yields each JSON object."""
    with open(file_path, 'r') as f:
        for line in f:
            try:
                obj = json.loads(line)
                yield obj
            except json.JSONDecodeError:
                print(f"Skipping invalid JSON line: {line.strip()}")



In [13]:
# Upload schema file
schema_filename = "gcs_json_schema.json"
with open(os.path.join(schema_filename), "r") as infile:
    gcs_schema = json.load(infile)

# gcs_schema

# Read JSONL data into a list
for file in os.listdir(local_output_path):
    if file != schema_filename:
        print("Testing file: {} ".format(file))
        jsonl_lists = []
        for item in read_jsonl_file_json(os.path.join(local_output_path, file)):
            jsonl_lists.append(item)

        for item_line in jsonl_lists:
            jsonschema.validate(instance=item_line,
                                schema=gcs_schema)



Testing file: ccrctccolumbiaedu_2025May01_text.jsonl 
Testing file: laocagov_2025May01_text.jsonl 
Testing file: wwwecsorg_2025May01_text.jsonl 
Testing file: icangotocollegecom_2025May01_text.jsonl 
Testing file: nscresearchcenterorg_2025May01_text.jsonl 
Testing file: wwwccleagueorg_2025May01_text.jsonl 
Testing file: calmattersdigitaldemocracyorg_2025May01_text.jsonl 
Testing file: enwikipediaorg_2025May01_text.jsonl 
Testing file: wwwaaccncheedu_2025May01_text.jsonl 
Testing file: cccaoeorg_2025May01_text.jsonl 
Testing file: wwwccccoedu_2025May01_text.jsonl 


## Upload JSONL files to GCP

In [10]:
gcs_path = "crawl_data/jsonl_files2"
local_output_path = "../data/jsonl_files"

gct.upload_directory_to_gcs(local_directory=local_output_path,
                            gcs_project_id=os.environ["GOOGLE_CLOUD_PROJECT"],
                            gcs_bucket_name=gcs_bucket_name,
                            gcs_directory=gcs_path)



Uploaded ../data/jsonl_files/ccrctccolumbiaedu_2025May01_text.jsonl to gs://ccc-crawl_data_xb/crawl_data/jsonl_files2/ccrctccolumbiaedu_2025May01_text.jsonl
Uploaded ../data/jsonl_files/laocagov_2025May01_text.jsonl to gs://ccc-crawl_data_xb/crawl_data/jsonl_files2/laocagov_2025May01_text.jsonl
Uploaded ../data/jsonl_files/wwwecsorg_2025May01_text.jsonl to gs://ccc-crawl_data_xb/crawl_data/jsonl_files2/wwwecsorg_2025May01_text.jsonl
Uploaded ../data/jsonl_files/icangotocollegecom_2025May01_text.jsonl to gs://ccc-crawl_data_xb/crawl_data/jsonl_files2/icangotocollegecom_2025May01_text.jsonl
Uploaded ../data/jsonl_files/nscresearchcenterorg_2025May01_text.jsonl to gs://ccc-crawl_data_xb/crawl_data/jsonl_files2/nscresearchcenterorg_2025May01_text.jsonl
Uploaded ../data/jsonl_files/wwwccleagueorg_2025May01_text.jsonl to gs://ccc-crawl_data_xb/crawl_data/jsonl_files2/wwwccleagueorg_2025May01_text.jsonl
Uploaded ../data/jsonl_files/calmattersdigitaldemocracyorg_2025May01_text.jsonl to gs://cc